In [1]:
##modules
#%matplotlib widget
#%matplotlib inline
#
%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

#matplotlib.use('TkAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs

from statsmodels.tsa.stattools import acf
import numpy as np

import pandas as pd

In [2]:


#subj = sys.argv[1] ## name of participant list
# Variables path
layer_script = "block"

#process=""
disco="g"

# Carpeta general
datadir = Path(f"{disco}:\MOUS_204")

#carpetas generales de datos
# mri_dir = datadir / f"{subj}"/"anat"
# meg_dir = datadir / f"{subj}"/"meg"

# Carpeta de preprocesado
output_preproc = datadir / "output_preproc"

channels_structure_path = output_preproc / "channels_structure"


preproc_path = output_preproc / f"preproc_{layer_script}"
preproc_path.mkdir(parents=True, exist_ok=True)
 
# Carpeta de epocas "sucias"
epochs_path = preproc_path / f"epochs_{layer_script}"
epochs_path.mkdir(parents=True, exist_ok=True) 

# Carpeta de ICA
ICA_path = preproc_path / f"ICA_{layer_script}"
ICA_path.mkdir(parents=True, exist_ok=True)

# Épocas limpias
epochs_clean_path = preproc_path / f"epochs_clean_{layer_script}"
epochs_clean_path.mkdir(parents=True, exist_ok=True)

#epocas evoked
evoked_path = Path(preproc_path) / f"evoked_{layer_script}"
evoked_path.mkdir(parents=True, exist_ok=True)
 

# Definir la carpeta de output_source antes de usarla
output_source = Path(r"g:\MOUS_204\output_source")

source_path = output_source / f"source_{layer_script}"
source_path.mkdir(parents=True, exist_ok=True)

#raw_hsp es el raw con fiducials cargados
raw_hsp_path = source_path / f"raw_hsp"
raw_hsp_path.mkdir(parents=True, exist_ok=True)

# Carpeta de forward solution
fwd_path = source_path / f"fwd"
fwd_path.mkdir(parents=True, exist_ok=True)

# Carpeta de inverse solution
inverse_path = source_path / f"inverse"
inverse_path.mkdir(parents=True, exist_ok=True)

output_analysis = datadir / "output_analysis"
analysis_path = output_analysis / f"analysis_{layer_script}"
analysis_path.mkdir(parents=True, exist_ok=True)

ACW_path = analysis_path / f"acw_{layer_script}"
ACW_path.mkdir(parents=True, exist_ok=True)

PLE_path = analysis_path / f"PLE_{layer_script}"
PLE_path.mkdir(parents=True, exist_ok=True)

ISC_path = analysis_path / f"ISC_{layer_script}"
ISC_path.mkdir(parents=True, exist_ok=True)

mne.utils.set_config('SUBJECTS_DIR', r'\\wsl$\Ubuntu-20.04\usr\local\freesurfer\subjects', set_env=True)

<>:9: SyntaxWarning: invalid escape sequence '\M'
<>:9: SyntaxWarning: invalid escape sequence '\M'
C:\Users\UCM\AppData\Local\Temp\ipykernel_12824\1068408573.py:9: SyntaxWarning: invalid escape sequence '\M'
  datadir = Path(f"{disco}:\MOUS_204")


In [3]:
channels = pd.read_csv(channels_structure_path / "channels_mag.csv")
channels_mag=channels[channels["canal_efectivo"].notna()]["canal_efectivo"]

channels_mag=channels_mag.tolist()
print(channels_mag)
indice_channels_efectivos = channels[channels["canal_efectivo"].notna()]["indice"]
indice_channels_efectivos=indice_channels_efectivos.tolist()
# del channels

['MLC12-4304', 'MLC13-4304', 'MLC14-4304', 'MLC15-4304', 'MLC16-4304', 'MLC17-4304', 'MLC21-4304', 'MLC22-4304', 'MLC23-4304', 'MLC24-4304', 'MLC25-4304', 'MLC31-4304', 'MLC32-4304', 'MLC41-4304', 'MLC42-4304', 'MLC51-4304', 'MLC52-4304', 'MLC53-4304', 'MLC54-4304', 'MLC55-4304', 'MLC61-4304', 'MLC62-4304', 'MLC63-4304', 'MLF11-4304', 'MLF12-4304', 'MLF13-4304', 'MLF14-4304', 'MLF21-4304', 'MLF22-4304', 'MLF23-4304', 'MLF24-4304', 'MLF25-4304', 'MLF31-4304', 'MLF32-4304', 'MLF33-4304', 'MLF34-4304', 'MLF35-4304', 'MLF41-4304', 'MLF42-4304', 'MLF43-4304', 'MLF44-4304', 'MLF45-4304', 'MLF46-4304', 'MLF51-4304', 'MLF52-4304', 'MLF53-4304', 'MLF54-4304', 'MLF55-4304', 'MLF56-4304', 'MLF61-4304', 'MLF62-4304', 'MLF63-4304', 'MLF64-4304', 'MLF65-4304', 'MLF66-4304', 'MLF67-4304', 'MLO11-4304', 'MLO12-4304', 'MLO13-4304', 'MLO14-4304', 'MLO21-4304', 'MLO22-4304', 'MLO23-4304', 'MLO24-4304', 'MLO31-4304', 'MLO32-4304', 'MLO33-4304', 'MLO34-4304', 'MLO41-4304', 'MLO42-4304', 'MLO43-4304', 'MLO4

In [4]:
#lectura de acw
acw_50_df = pd.read_pickle(ACW_path /f"acw_50_df.pickle")


In [5]:
##creacion dataframe ACW_50
# acw_all = pd.read_pickle(ACW_path /f"autocorrelation_subjects_all.pickle")
# acw_50_df = acw_all[["Subject", "Condition", "Epoch",
#                    "Elect", "acw_50_elect_all_epoch_all"]]
# acw_50_df.to_pickle(ACW_path / f"acw_50_df.pickle")

# acw_0_df = acw_all[["Subject", "Condition", "Epoch",
#                    "Elect", "acw_0_elect_all_epoch_all"]]
# acw_0_df.to_pickle(ACW_path / f"acw_0_df.pickle")

# del acw_all, del acw_0_df

In [5]:
##valores de las columnas
condition = acw_50_df["Condition"].unique()
print("Condiciones en los datos:", condition)

subjects = acw_50_df["Subject"].unique()
print("Sujetos en los datos:", subjects)

elect_all= acw_50_df["Elect"].unique()
print("sensores en los datos:", elect_all)

epochs_all= acw_50_df["Epoch"].unique()
print("Epochs en los datos:", epochs_all)


Condiciones en los datos: ['zinnen' 'woorden']
Sujetos en los datos: ['sub-A2002' 'sub-A2003' 'sub-A2004' 'sub-A2005' 'sub-A2006' 'sub-A2007'
 'sub-A2008' 'sub-A2009' 'sub-A2010' 'sub-A2013' 'sub-A2014' 'sub-A2015'
 'sub-A2016' 'sub-A2017' 'sub-A2019' 'sub-A2020' 'sub-A2021' 'sub-A2024'
 'sub-A2025']
sensores en los datos: ['MLC12-4304' 'MLC13-4304' 'MLC14-4304' 'MLC15-4304' 'MLC16-4304'
 'MLC17-4304' 'MLC21-4304' 'MLC22-4304' 'MLC23-4304' 'MLC24-4304'
 'MLC25-4304' 'MLC31-4304' 'MLC32-4304' 'MLC41-4304' 'MLC42-4304'
 'MLC51-4304' 'MLC52-4304' 'MLC53-4304' 'MLC54-4304' 'MLC55-4304'
 'MLC61-4304' 'MLC62-4304' 'MLC63-4304' 'MLF11-4304' 'MLF12-4304'
 'MLF13-4304' 'MLF14-4304' 'MLF21-4304' 'MLF22-4304' 'MLF23-4304'
 'MLF24-4304' 'MLF25-4304' 'MLF31-4304' 'MLF32-4304' 'MLF33-4304'
 'MLF34-4304' 'MLF35-4304' 'MLF41-4304' 'MLF42-4304' 'MLF43-4304'
 'MLF44-4304' 'MLF45-4304' 'MLF46-4304' 'MLF51-4304' 'MLF52-4304'
 'MLF53-4304' 'MLF54-4304' 'MLF55-4304' 'MLF56-4304' 'MLF61-4304'
 'MLF62-4304' '

In [6]:
#lectura de los diccionarios ISC_zinnen y ISC_woorden
dict_woorden_block = pd.read_pickle(ISC_path /f"dict_woorden_block.pkl")
dict_zinnen_block = pd.read_pickle(ISC_path /f"dict_zinnen_block.pkl")

In [7]:
##channels significant adjusted in each condition
numbers_channels_woorden= dict_woorden_block["significant_channels_adjusted_woorden"]
numbers_channels_zinnen= dict_zinnen_block["significant_channels_adjusted_zinnen"]
print("numbers_channels_woorden", numbers_channels_woorden, "len",len(numbers_channels_woorden))

print("numbers_channels_zinnen", numbers_channels_zinnen, "len", len(numbers_channels_zinnen))


numbers_channels_woorden [  7  13  14  25  26  30  31  34  35  36  39  40  41  42  44  45  46  47
  48  54  71  79  83  84  87  88  89  94  95  97  98  99 101 103 104 105
 106 109 110 111 112 113 114 115 116 117 118 119 120 122 123 124 125 126
 127 128 129 131 134 136 137 139 140 141 142 143 144 145 146 148 152 153
 158 174 186 187 191 200 202 204 206 209 217 218 219 221 223 225 226 227
 228 229 231 233 234 235 236 240 241 242 243 244 247 248 249 250 251 252
 256 257 258 259 260] len 113
numbers_channels_zinnen [  1   2   3   4   6   7   8   9  10  11  13  14  16  17  18  19  20  21
  22  23  24  25  26  27  28  29  30  31  32  33  35  36  40  41  42  47
  48  51  53  54  55  59  67  68  70  74  75  76  77  78  81  82  83  84
  85  86  87  88  89  90  91  92  93  94  95  96  97  98  99 101 103 104
 105 106 109 110 111 112 113 114 117 118 119 120 121 122 123 124 125 126
 127 128 129 130 135 139 140 143 144 145 146 148 149 150 151 152 153 154
 155 157 158 161 162 163 164 166 167 168 170 

In [8]:
##channels significant adjusted in each condition
numbers_channels_woorden= dict_woorden_block["significant_channels_adjusted_woorden"]
numbers_channels_zinnen= dict_zinnen_block["significant_channels_adjusted_zinnen"]
print("numbers_channels_woorden", numbers_channels_woorden, "numbers_channels_zinnen", numbers_channels_zinnen)
print("len(numbers_channels_woorden)",len(numbers_channels_woorden),  "len(numbers_channels_zinnen)",len(numbers_channels_zinnen))

names_channels_woorden=[channels_mag[i] for i in numbers_channels_woorden]
names_channels_zinnen=[channels_mag[i] for i in numbers_channels_zinnen]

print("names_channels_woorden", names_channels_woorden)
print("names_channels_zinnen", names_channels_zinnen)

#establecimiento de canales palabras, canales frases y canales mixtos
numbers_channels_only_woorden = list(
    set(numbers_channels_woorden) - set(numbers_channels_zinnen)
)

numbers_channels_only_zinnen = list(
    set(numbers_channels_zinnen) - set(numbers_channels_woorden)
)

number_channels_intersection = list(
    set(numbers_channels_woorden) & set(numbers_channels_zinnen)
)

print(f"len(significant_channels_only_woorden): {len(numbers_channels_only_woorden)},\
      len(significant_channels_only_zinnen): {len(numbers_channels_only_zinnen)},\
      len(significant_channels_intersection) {len(number_channels_intersection)}")

#create list of names of channels

names_channels_only_woorden = [channels_mag[i] for i in numbers_channels_only_woorden]
names_channels_only_zinnen = [channels_mag[i] for i in numbers_channels_only_zinnen]
names_channels_intersection = [channels_mag[i] for i in number_channels_intersection]

numbers_channels_woorden [  7  13  14  25  26  30  31  34  35  36  39  40  41  42  44  45  46  47
  48  54  71  79  83  84  87  88  89  94  95  97  98  99 101 103 104 105
 106 109 110 111 112 113 114 115 116 117 118 119 120 122 123 124 125 126
 127 128 129 131 134 136 137 139 140 141 142 143 144 145 146 148 152 153
 158 174 186 187 191 200 202 204 206 209 217 218 219 221 223 225 226 227
 228 229 231 233 234 235 236 240 241 242 243 244 247 248 249 250 251 252
 256 257 258 259 260] numbers_channels_zinnen [  1   2   3   4   6   7   8   9  10  11  13  14  16  17  18  19  20  21
  22  23  24  25  26  27  28  29  30  31  32  33  35  36  40  41  42  47
  48  51  53  54  55  59  67  68  70  74  75  76  77  78  81  82  83  84
  85  86  87  88  89  90  91  92  93  94  95  96  97  98  99 101 103 104
 105 106 109 110 111 112 113 114 117 118 119 120 121 122 123 124 125 126
 127 128 129 130 135 139 140 143 144 145 146 148 149 150 151 152 153 154
 155 157 158 161 162 163 164 166 167 168 170 171 172 

## Posibles comparaciones estadísticas
len(significant_channels_only_woorden): 25, len(significant_channels_only_zinnen): 92, len(significant_channels_intersection) 88

Primer problema, el numero de canales zinnen es claramente superior al de woorden, big problem, aunque la interseccion es alta, tal como esperaríamos. Seguramente esto sea porque hay ruido 

- comparación ACW-50 en zinnen entre estos canales zinnen y canales woorden  + comapracion en woorden de lo mismo

- comparación entre solo zinnen con intersection  en woorden y en zinnen (aunque ojo, esto es básicamente aceptar que los canales solo Zinnen son ruido)

- comapración en el promedio general de acw en ambas condiciones de los canalaes zinnen vs promedio general de canales woorden

- comparación de la intersección vs canales no signficativos: no sé muy bien como interpretar esto

- 

In [9]:
channels_type=["names_channels_only_woorden", "names_channels_only_zinnen", "names_channels_intersection"]



##matrices de valores de acw_50, divididas por condicion y tipo de channel
X_zinnen_ch_zinnen = []
X_zinnen_ch_woorden = []
X_zinnen_ch_intersection = []

X_woorden_ch_zinnen = []
X_woorden_ch_woorden = []
X_woorden_ch_intersection = []

##Filtras la condicion
for cond in condition:
    acw_50_condition_df=acw_50_df[acw_50_df["Condition"] == f"{cond}"]

    #filtras por tipo de canal only zinnen o only woorden
    for ch_type in channels_type:

        #bucle if para seleccionar el tipo de canal
        if ch_type == "names_channels_only_zinnen":
            elects=names_channels_only_zinnen
            print(ch_type, elects)
        if ch_type == "names_channels_only_woorden":
            elects=names_channels_only_woorden
            print(ch_type, elects)
        if ch_type == "names_channels_intersection":
            elects=names_channels_intersection
            print(ch_type, elects)

        

names_channels_only_woorden ['MRC11-4304', 'MRC14-4304', 'MRC16-4304', 'MRC17-4304', 'MRC24-4304', 'MRC25-4304', 'MLF33-4304', 'MLF43-4304', 'MLF52-4304', 'MLF53-4304', 'MLF54-4304', 'MRF67-4304', 'MRO11-4304', 'MRO21-4304', 'MLO44-4304', 'MRP12-4304', 'MLP23-4304', 'MRP43-4304', 'MRP45-4304', 'MRP56-4304', 'MRP57-4304', 'MRT15-4304', 'MRT24-4304', 'MLT36-4304', 'MLT37-4304']
names_channels_only_zinnen ['MLC13-4304', 'MLC14-4304', 'MLC15-4304', 'MLC16-4304', 'MLC21-4304', 'MLC23-4304', 'MLC24-4304', 'MLC25-4304', 'MLC31-4304', 'MLC52-4304', 'MLC53-4304', 'MLC54-4304', 'MLC55-4304', 'MLC61-4304', 'MLC62-4304', 'MLC63-4304', 'MLF11-4304', 'MLF12-4304', 'MLF21-4304', 'MLF22-4304', 'MLF23-4304', 'MLF31-4304', 'MLF32-4304', 'MLF63-4304', 'MLF65-4304', 'MLF67-4304', 'MLO14-4304', 'MLO34-4304', 'MLO41-4304', 'MLO43-4304', 'MLO53-4304', 'MLP11-4304', 'MLP12-4304', 'MLP21-4304', 'MLP22-4304', 'MLP32-4304', 'MLP33-4304', 'MLP41-4304', 'MLP42-4304', 'MLP51-4304', 'MLP52-4304', 'MLP53-4304', 'MLP5

In [10]:
acw_50_condition_ch_type_df=acw_50_condition_df[acw_50_condition_df["Elect"].isin(elects)]


In [11]:
##selección de valores


channels_type=["names_channels_only_woorden", "names_channels_only_zinnen", "names_channels_intersection"]

##matrices de valores de acw_50, divididas por condicion y tipo de channel
X_zinnen_ch_zinnen = []
X_zinnen_ch_woorden = []
X_zinnen_ch_intersection = []

X_woorden_ch_zinnen = []
X_woorden_ch_woorden = []
X_woorden_ch_intersection = []

##Filtras la condicion
for cond in condition:
    acw_50_condition_df=acw_50_df[acw_50_df["Condition"] == f"{cond}"]

    #filtras por tipo de canal only zinnen o only woorden
    for ch_type in channels_type:

        #bucle if para seleccionar el tipo de canal
        if ch_type == "names_channels_only_zinnen":
            elects=names_channels_only_zinnen
        if ch_type == "names_channels_only_woorden":
            elects=names_channels_only_woorden
        if ch_type == "names_channels_intersection":
            elects=names_channels_intersection

        acw_50_condition_ch_type_df=acw_50_condition_df[acw_50_condition_df["Elect"].isin(elects)]
        #filtras por sujeto 
        for subj in subjects:
                #creacion de lista de valores de acw_50 para cada sujeto
                acw_50_epoch_list= []

                #filtras por sujeto
                for epoch in epochs_all:
                    # Extraer el valor de ACW_50 para la combinación actual
                    try:
                    #coges el valor de acw_50 para el sujeto y el epoch
                        print(f"subj:{subj},epoch, {epoch}")
                        acw_50_elect_all_epoch_all = acw_50_condition_ch_type_df[(acw_50_condition_ch_type_df["Subject"] == subj) & (acw_50_condition_ch_type_df["Epoch"] == epoch)]["acw_50_elect_all_epoch_all"]
                        if not acw_50_elect_all_epoch_all.empty:
                            acw_50_epoch_list.append(acw_50_elect_all_epoch_all)

                    except Exception as e:
                        print(e, "probablemente faltaban epochs en algunos sujetos")
                        continue
                # Convertir la lista a un array de numpy
                acw_50_epoch_array = np.array(acw_50_epoch_list)
                #take the mean on epochs
                acw_50_epoch_mean = np.mean(acw_50_epoch_array, axis=0)

                #add it to the different lists
                if cond == "zinnen":
                    if ch_type == "names_channels_only_zinnen":
                        X_zinnen_ch_zinnen.append(acw_50_epoch_mean)
                    elif ch_type == "names_channels_only_woorden":
                        X_zinnen_ch_woorden.append(acw_50_epoch_mean)
                    elif ch_type == "names_channels_intersection":
                        X_zinnen_ch_intersection.append(acw_50_epoch_mean)
                if cond == "woorden":
                    if ch_type == "names_channels_only_zinnen":
                        X_woorden_ch_zinnen.append(acw_50_epoch_mean)
                    elif ch_type == "names_channels_only_woorden":
                        X_woorden_ch_woorden.append(acw_50_epoch_mean)
                    elif ch_type == "names_channels_intersection":
                        X_woorden_ch_intersection.append(acw_50_epoch_mean)


X_zinnen_ch_zinnen = np.array(X_zinnen_ch_zinnen)
X_zinnen_ch_woorden = np.array(X_zinnen_ch_woorden)
X_zinnen_ch_intersection = np.array(X_zinnen_ch_intersection)

X_woorden_ch_zinnen = np.array(X_woorden_ch_zinnen)
X_woorden_ch_woorden = np.array(X_woorden_ch_woorden)
X_woorden_ch_intersection = np.array(X_woorden_ch_intersection)

print("Shapes de las matrices de valores ACW_50:\n")

print("▶ Condición: ZINNEN")
print("  - Canales only_zinnen      :", X_zinnen_ch_zinnen.shape)
print("  - Canales only_woorden     :", X_zinnen_ch_woorden.shape)
print("  - Canales intersection     :", X_zinnen_ch_intersection.shape)

print("\n▶ Condición: WOORDEN")
print("  - Canales only_zinnen      :", X_woorden_ch_zinnen.shape)
print("  - Canales only_woorden     :", X_woorden_ch_woorden.shape)
print("  - Canales intersection     :", X_woorden_ch_intersection.shape)


subj:sub-A2002,epoch, 0
subj:sub-A2002,epoch, 1
subj:sub-A2002,epoch, 2
subj:sub-A2002,epoch, 3
subj:sub-A2002,epoch, 4
subj:sub-A2002,epoch, 5
subj:sub-A2002,epoch, 6
subj:sub-A2002,epoch, 7
subj:sub-A2002,epoch, 8
subj:sub-A2002,epoch, 9
subj:sub-A2002,epoch, 10
subj:sub-A2002,epoch, 11
subj:sub-A2002,epoch, 12
subj:sub-A2002,epoch, 13
subj:sub-A2002,epoch, 14
subj:sub-A2002,epoch, 15
subj:sub-A2002,epoch, 16
subj:sub-A2002,epoch, 17
subj:sub-A2002,epoch, 18
subj:sub-A2002,epoch, 19
subj:sub-A2002,epoch, 20
subj:sub-A2002,epoch, 21
subj:sub-A2002,epoch, 22
subj:sub-A2002,epoch, 23
subj:sub-A2003,epoch, 0
subj:sub-A2003,epoch, 1
subj:sub-A2003,epoch, 2
subj:sub-A2003,epoch, 3
subj:sub-A2003,epoch, 4
subj:sub-A2003,epoch, 5
subj:sub-A2003,epoch, 6
subj:sub-A2003,epoch, 7
subj:sub-A2003,epoch, 8
subj:sub-A2003,epoch, 9
subj:sub-A2003,epoch, 10
subj:sub-A2003,epoch, 11
subj:sub-A2003,epoch, 12
subj:sub-A2003,epoch, 13
subj:sub-A2003,epoch, 14
subj:sub-A2003,epoch, 15
subj:sub-A2003,epoch

In [12]:

## permutaciones para comparar los valores de acw_50 entre los canales significativos de cada condicion
X_zinnen_ch_zinnen_mean = X_zinnen_ch_zinnen.mean(axis=1)
X_zinnen_ch_woorden_mean = X_zinnen_ch_woorden.mean(axis=1)
print(f"X_zinnen_ch_zinnen_mean, {X_zinnen_ch_zinnen_mean.shape}")	
print(f"X_zinnen_ch_woorden_mean, {X_zinnen_ch_woorden_mean.shape}")	



X_woorden_ch_zinnen_mean = X_woorden_ch_zinnen.mean(axis=1)
X_woorden_ch_woorden_mean = X_woorden_ch_woorden.mean(axis=1)
print(f"X_woorden_ch_zinnen_mean, {X_woorden_ch_zinnen_mean.shape}")	
print(f"X_woorden_ch_woorden_mean, {X_woorden_ch_woorden_mean.shape}")


X_zinnen_ch_intersection_mean = X_zinnen_ch_intersection.mean(axis=1)
X_woorden_ch_intersection_mean = X_woorden_ch_intersection.mean(axis=1)

print(f"X_zinnen_ch_intersection_mean, {X_zinnen_ch_intersection_mean.shape}")
print(f"X_woorden_ch_intersection_mean, {X_woorden_ch_intersection_mean.shape}")


# Crear el diccionario con nombres descriptivos
data_dict = {
    "X_zinnen_ch_zinnen_mean": X_zinnen_ch_zinnen_mean,
    "X_zinnen_ch_woorden_mean": X_zinnen_ch_woorden_mean,
    "X_woorden_ch_zinnen_mean": X_woorden_ch_zinnen_mean,
    "X_woorden_ch_woorden_mean": X_woorden_ch_woorden_mean,
    "X_zinnen_ch_intersection_mean": X_zinnen_ch_intersection_mean,
    "X_woorden_ch_intersection_mean": X_woorden_ch_intersection_mean,
}

X_zinnen_ch_zinnen_mean, (19,)
X_zinnen_ch_woorden_mean, (19,)
X_woorden_ch_zinnen_mean, (19,)
X_woorden_ch_woorden_mean, (19,)
X_zinnen_ch_intersection_mean, (19,)
X_woorden_ch_intersection_mean, (19,)


# Comparaciones Zinnen only vs intersection


In [13]:

from scipy.stats import permutation_test

##dependent condition
def paired_statistic(diff, _):
    return np.mean(diff)


# Independent condition
def diff_means(x, y):
    return np.mean(x) - np.mean(y)

# if we assume dependency

In [14]:


experimental_condition=["zinnen", "woorden"]
type_channel=["intersection_mean", "woorden_mean"]
print("DEPENDENT analysis")

for exp in experimental_condition:
    print(f"in {exp} condition")
    for ch in type_channel:
        print(f"for differences between zinnen only and {ch}:")
        # print(f"Processing {exp} for differences between zinenn and {ch}...")
        ##notice that in this comparison X is alwas ch_zinnnen_mean
        x= data_dict[f"X_{exp}_ch_zinnen_mean"]
        ##notice that in this comparison y is  ch_woorden_mean or ch_intersection_mean
        y=data_dict[f"X_{exp}_ch_{ch}"]

        diff=x-y
        # Permutation test
        res = permutation_test(
            (diff, np.zeros_like(diff)),
            #
            statistic=paired_statistic,
            vectorized=False,
            n_resamples=10000,
            alternative='greater', 
            random_state=42
        )

        # Cohen's d
        mean_diff = np.mean(diff)
        std_diff = np.std(diff, ddof=1)
        cohens_d = mean_diff / std_diff

        print(f"   Statistic value: {res.statistic:.6f}")
        print(f"   p-value        : {res.pvalue:.5f}")
        print(f"   Cohen's d      : {cohens_d:.3f}")

DEPENDENT analysis
in zinnen condition
for differences between zinnen only and intersection_mean:
   Statistic value: -0.003315
   p-value        : 1.00000
   Cohen's d      : -1.089
for differences between zinnen only and woorden_mean:
   Statistic value: -0.000452
   p-value        : 0.99780
   Cohen's d      : -0.623
in woorden condition
for differences between zinnen only and intersection_mean:
   Statistic value: -0.003189
   p-value        : 1.00000
   Cohen's d      : -1.284
for differences between zinnen only and woorden_mean:
   Statistic value: -0.000634
   p-value        : 0.99950
   Cohen's d      : -0.777


In [15]:
##if we assume independency


##comapraciones en la condicion zinnen
experimental_condition=["zinnen", "woorden"]
type_channel=["intersection_mean", "woorden_mean"]

print("INDEPENDENT analysis")
for exp in experimental_condition:
    print(f"in {exp} condition")
    for ch in type_channel:
        print(f"for differences between zinenn and {ch}:")
        # print(f"Processing {exp} for differences between zinenn and {ch}...")
        ##notice that in this comparison X is alwas ch_zinnnen_mean
        x= data_dict[f"X_{exp}_ch_zinnen_mean"]
        ##notice that in this comparison y is  ch_woorden_mean or ch_intersection_mean
        y=data_dict[f"X_{exp}_ch_{ch}"]

        # Hacemos la prueba de permutación
        res = permutation_test(
            (x, y),
            statistic=diff_means,
            vectorized=False,
            n_resamples=10000,
            alternative='greater', 
            random_state=42
        )

        # Calcular Cohen's d (independent)
        mean_x, mean_y = np.mean(x), np.mean(y)
        std_x, std_y = np.std(x, ddof=1), np.std(y, ddof=1)
        n_x, n_y = len(x), len(y)

        pooled_std = np.sqrt(((n_x - 1) * std_x**2 + (n_y - 1) * std_y**2) / (n_x + n_y - 2))
        cohens_d = (mean_x - mean_y) / pooled_std

        # Imprimir resultados
        print(f"   Statistic value: {res.statistic:.6f}")
        print(f"   p-value        : {res.pvalue:.5f}")
        print(f"   Cohen's d      : {cohens_d:.3f}")


INDEPENDENT analysis
in zinnen condition
for differences between zinenn and intersection_mean:
   Statistic value: -0.003315
   p-value        : 0.98160
   Cohen's d      : -0.702
for differences between zinenn and woorden_mean:
   Statistic value: -0.000452
   p-value        : 0.65363
   Cohen's d      : -0.132
in woorden condition
for differences between zinenn and intersection_mean:
   Statistic value: -0.003189
   p-value        : 0.98910
   Cohen's d      : -0.776
for differences between zinenn and woorden_mean:
   Statistic value: -0.000634
   p-value        : 0.72693
   Cohen's d      : -0.205


## using average in both conditions

In [16]:

experimental_condition=["zinnen", "woorden"]
type_channel=["intersection_mean", "woorden_mean"]
print("DEPENDENT analysis")


for ch in type_channel:
    print(f"for differences between zinnen only and {ch}:")
    # print(f"Processing {exp} for differences between zinenn and {ch}...")
    ##notice that in this comparison X is always ch_zinnnen_mean
    x= (data_dict[f"X_zinnen_ch_zinnen_mean"] + data_dict[f"X_woorden_ch_zinnen_mean"])/2

    ##notice that in this comparison y is  ch_woorden_mean or ch_intersection_mean
    y=(data_dict[f"X_zinnen_ch_{ch}"] + data_dict[f"X_woorden_ch_{ch}"])/2

    diff=x-y
    # Permutation test
    res = permutation_test(
        (diff, np.zeros_like(diff)),
        #
        statistic=paired_statistic,
        vectorized=False,
        n_resamples=10000,
        alternative='greater', 
        random_state=42
    )
    # Cohen's d
    mean_diff = np.mean(diff)
    std_diff = np.std(diff, ddof=1)
    cohens_d = mean_diff / std_diff

    print(f"   Statistic value: {res.statistic:.6f}")
    print(f"   p-value        : {res.pvalue:.5f}")
    print(f"   Cohen's d      : {cohens_d:.3f}")

DEPENDENT analysis
for differences between zinnen only and intersection_mean:
   Statistic value: -0.003252
   p-value        : 1.00000
   Cohen's d      : -1.192
for differences between zinnen only and woorden_mean:
   Statistic value: -0.000543
   p-value        : 0.99930
   Cohen's d      : -0.721


In [17]:

experimental_condition=["zinnen", "woorden"]
type_channel=["intersection_mean", "woorden_mean"]
print("INdependent analysis")


for ch in type_channel:
    print(f"for differences between zinnen only and {ch}:")
    # print(f"Processing {exp} for differences between zinenn and {ch}...")
    ##notice that in this comparison X is always ch_zinnnen_mean
    x= (data_dict[f"X_zinnen_ch_zinnen_mean"] + data_dict[f"X_woorden_ch_zinnen_mean"])/2

    ##notice that in this comparison y is  ch_woorden_mean or ch_intersection_mean
    y=(data_dict[f"X_zinnen_ch_{ch}"] + data_dict[f"X_woorden_ch_{ch}"])/2

    diff=x-y
    # Permutation test
    res = permutation_test(
        (diff, np.zeros_like(diff)),
        #
        statistic=diff_means,
        vectorized=False,
        n_resamples=10000,
        alternative='greater', 
        random_state=42
    )

    # Calcular Cohen's d (independent)
    mean_x, mean_y = np.mean(x), np.mean(y)
    std_x, std_y = np.std(x, ddof=1), np.std(y, ddof=1)
    n_x, n_y = len(x), len(y)

    pooled_std = np.sqrt(((n_x - 1) * std_x**2 + (n_y - 1) * std_y**2) / (n_x + n_y - 2))
    cohens_d = (mean_x - mean_y) / pooled_std

    # Imprimir resultados
    print(f"   Statistic value: {res.statistic:.6f}")
    print(f"   p-value        : {res.pvalue:.5f}")
    print(f"   Cohen's d      : {cohens_d:.3f}")

INdependent analysis
for differences between zinnen only and intersection_mean:
   Statistic value: -0.003252
   p-value        : 1.00000
   Cohen's d      : -0.738
for differences between zinnen only and woorden_mean:
   Statistic value: -0.000543
   p-value        : 0.99930
   Cohen's d      : -0.167


# using flatten

please note thet this is wrong

In [19]:
# ## permutaciones para comparar los valores de acw_50 entre los canales significativos de cada condicion
# X_zinnen_ch_zinnen_flatten = X_zinnen_ch_zinnen.flatten(axis=1)
# X_zinnen_ch_woorden_flatten = X_zinnen_ch_woorden.flatten(axis=1)
# print(f"X_zinnen_ch_zinnen_flatten, {X_zinnen_ch_zinnen_flatten.shape}")	
# print(f"X_zinnen_ch_woorden_flatten, {X_zinnen_ch_woorden_flatten.shape}")	



# X_woorden_ch_zinnen_flatten = X_woorden_ch_zinnen.flatten(axis=1)
# X_woorden_ch_woorden_flatten = X_woorden_ch_woorden.flatten(axis=1)
# print(f"X_woorden_ch_zinnen_flatten, {X_woorden_ch_zinnen_flatten.shape}")	
# print(f"X_woorden_ch_woorden_flatten, {X_woorden_ch_woorden_flatten.shape}")


# X_zinnen_ch_intersection_flatten = X_zinnen_ch_intersection.flatten(axis=1)
# X_woorden_ch_intersection_flatten = X_woorden_ch_intersection.flatten(axis=1)

# print(f"X_zinnen_ch_intersection_flatten, {X_zinnen_ch_intersection_flatten.shape}")
# print(f"X_woorden_ch_intersection_flatten, {X_woorden_ch_intersection_flatten.shape}")


# # Crear el diccionario con nombres descriptivos
# data_dict = {
#     "X_zinnen_ch_zinnen_flatten": X_zinnen_ch_zinnen_flatten,
#     "X_zinnen_ch_woorden_flatten": X_zinnen_ch_woorden_flatten,
#     "X_woorden_ch_zinnen_flatten": X_woorden_ch_zinnen_flatten,
#     "X_woorden_ch_woorden_flatten": X_woorden_ch_woorden_flatten,
#     "X_zinnen_ch_intersection_flatten": X_zinnen_ch_intersection_flatten,
#     "X_woorden_ch_intersection_flatten": X_woorden_ch_intersection_flatten}

# comparison of word sentence channels in both conditions

Before we compared differences of word and sentence channels. Now we are going to compare the channels with themselves in different experimental conditions

In [18]:
## statistical test

## REVIEWW THISSS
experimental_condition=["zinnen", "woorden"]
type_channel=["zinnen_mean","intersection_mean", "woorden_mean"]
print("DEPENDENT analysis")


for ch in type_channel:
    print(f"for differences in ch {ch}  between zinnen and word condition:")
    # print(f"Processing {exp} for differences between zinenn and {ch}...")

    #ch value in zinnen condition
    x= data_dict[f"X_zinnen_ch_{ch}"]

    y=data_dict[f"X_woorden_ch_{ch}"]

    print(f"X is: X_zinnen_ch_{ch}", x.shape)
    print(f"Y is: X_woorden_ch_{ch}", y.shape)


    diff=x-y
    # Permutation test
    res = permutation_test(
        (diff, np.zeros_like(diff)),
        statistic=paired_statistic,
        vectorized=False,
        n_resamples=10000,
        alternative='greater', 
        random_state=42
    )

    # Calcular Cohen's d para muestras emparejadas
    mean_diff = np.mean(diff)
    std_diff = np.std(diff, ddof=1)
    cohens_d = mean_diff / std_diff

    # Imprimir resultados
    print(f"   Statistic value: {res.statistic:.6f}")
    print(f"   p-value        : {res.pvalue:.5f}")
    print(f"   Cohen's d      : {cohens_d:.3f}")

DEPENDENT analysis
for differences in ch zinnen_mean  between zinnen and word condition:
X is: X_zinnen_ch_zinnen_mean (19,)
Y is: X_woorden_ch_zinnen_mean (19,)
   Statistic value: 0.000397
   p-value        : 0.00020
   Cohen's d      : 0.835
for differences in ch intersection_mean  between zinnen and word condition:
X is: X_zinnen_ch_intersection_mean (19,)
Y is: X_woorden_ch_intersection_mean (19,)
   Statistic value: 0.000523
   p-value        : 0.01620
   Cohen's d      : 0.450
for differences in ch woorden_mean  between zinnen and word condition:
X is: X_zinnen_ch_woorden_mean (19,)
Y is: X_woorden_ch_woorden_mean (19,)
   Statistic value: 0.000215
   p-value        : 0.01310
   Cohen's d      : 0.507
